In [69]:
import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input,Dropout

import keras_tuner as kt

In [2]:
df = pd.read_csv(r"C:\Users\divya\OneDrive\DS\EDA\diabetes.csv")

In [3]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [4]:
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

In [5]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [6]:
X = scaler.fit_transform(X)

In [7]:
from sklearn.model_selection import train_test_split
X_train, X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [8]:
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense

In [9]:
model = Sequential()
model.add(Dense(32,activation='relu',input_dim=8))
model.add(Dense(1,activation='sigmoid'))

model.compile(optimizer='Adam',loss='binary_crossentropy',metrics=['accuracy'])

C:\Users\divya\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [10]:
model.fit(X_train,y_train, batch_size=32, epochs=10,validation_data=(X_test,y_test))

Epoch 1/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.4625 - loss: 0.7261 - val_accuracy: 0.6234 - val_loss: 0.6662
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5782 - loss: 0.6706 - val_accuracy: 0.6688 - val_loss: 0.6275
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6792 - loss: 0.6297 - val_accuracy: 0.6948 - val_loss: 0.5965
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7166 - loss: 0.5948 - val_accuracy: 0.7532 - val_loss: 0.5730
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7410 - loss: 0.5681 - val_accuracy: 0.7727 - val_loss: 0.5526
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7622 - loss: 0.5463 - val_accuracy: 0.7857 - val_loss: 0.5377
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7590 - loss: 0.5287 - val_accuracy: 0.7922 - val_loss: 0.5245
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7606 - loss: 0.5139 - val_accuracy: 0.7922 - val_loss

In [11]:
# 1. how to select appropriate optimizer
# 2. No, of nodes in a layer
# 3. how to select no. of layers
# 4. All in all one model

# 1. how to select appropriate optimizer

In [28]:
def build_model(hp):

    model = Sequential()

    model.add(Input(shape=(8,)))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))

    optimizer = hp.Choice(
        "optimizer",
        ["adam", "sgd", "rmsprop", "adadelta"]
    )

    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [12]:
model = build_model(kt.HyperParameters())

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_2 (Dense)                      │ (None, 32)                  │             288 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
tuner = kt.RandomSearch(
    hypermodel=build_model,
    objective="val_accuracy",
    max_trials=5,
    overwrite=True,
    directory="my_dir",
    project_name="optimizer_search"
)

In [14]:
tuner.search_space_summary()

Search space summary
Default search space size: 1
optimizer (Choice)
{'default': 'adam', 'conditions': [], 'values': ['adam', 'sgd', 'rmsprop', 'adadelta'], 'ordered': False}


In [15]:
keras.config.disable_traceback_filtering()

In [16]:
tuner.search(
    X_train,
    y_train,
    epochs=5,
    validation_data=(X_test, y_test),
    verbose=2
)

Trial 4 Complete [00h 00m 03s]
val_accuracy: 0.34415584802627563

Best val_accuracy So Far: 0.7792207598686218
Total elapsed time: 00h 00m 13s


In [17]:
print(tuner.oracle.trials)

{'0': <keras_tuner.src.engine.trial.Trial object at 0x000001F6170FA3C0>, '1': <keras_tuner.src.engine.trial.Trial object at 0x000001F6170B0F50>, '2': <keras_tuner.src.engine.trial.Trial object at 0x000001F619A5C2D0>, '3': <keras_tuner.src.engine.trial.Trial object at 0x000001F615D37CE0>}


In [18]:
tuner.results_summary()

Results summary
Results in my_dir\optimizer_search
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 2 summary
Hyperparameters:
optimizer: adam
Score: 0.7792207598686218

Trial 1 summary
Hyperparameters:
optimizer: rmsprop
Score: 0.6753246784210205

Trial 0 summary
Hyperparameters:
optimizer: sgd
Score: 0.6168830990791321

Trial 3 summary
Hyperparameters:
optimizer: adadelta
Score: 0.34415584802627563


In [19]:
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]

print(best_hp.values)

{'optimizer': 'adam'}


In [20]:
model = tuner.get_best_models(num_models=1)[0]

C:\Users\divya\anaconda3\Lib\site-packages\keras\src\saving\saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [21]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 32)                  │             288 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [22]:
model.fit(X_train,y_train,batch_size=32,epochs=100,initial_epoch=6,validation_data=(X_test,y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.7606 - loss: 0.4938 - val_accuracy: 0.7662 - val_loss: 0.5044
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7590 - loss: 0.4843 - val_accuracy: 0.7792 - val_loss: 0.5011
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7606 - loss: 0.4771 - val_accuracy: 0.7792 - val_loss: 0.4991
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7671 - loss: 0.4719 - val_accuracy: 0.7922 - val_loss: 0.4976
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7671 - loss: 0.4672 - val_accuracy: 0.7987 - val_loss: 0.4964
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7687 - loss: 0.4634 - val_accuracy: 0.7922 - val_loss: 0.4981
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7720 - loss: 0.4601 - val_accuracy: 0.7922 - val_loss: 0.4983
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7720 - loss: 0.4574 - val_accuracy: 0.78

# 2. No. of nodes in a layer

In [44]:
def build_model(hp):

    model = Sequential()

    units = hp.Int('units',8,128,step=8)

    model.add(Dense(units=units, activation='relu',input_dim=8))
    model.add(Dense(1,activation='sigmoid'))

    model.compile(optimizer='adam', loss='binary_crossentropy',metrics=['accuracy' ])

    return model

In [45]:
tuner = kt.RandomSearch(
    hypermodel=build_model,
    objective="val_accuracy",
    max_trials=5,
    overwrite=True,
    directory="my_dir",
    project_name="optimizer_search"
)

C:\Users\divya\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [46]:
tuner.search(
    X_train,
    y_train,
    epochs=5,
    validation_data=(X_test, y_test),
    verbose=2
)

Trial 5 Complete [00h 00m 04s]
val_accuracy: 0.7792207598686218

Best val_accuracy So Far: 0.7792207598686218
Total elapsed time: 00h 00m 20s


In [47]:
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]

print(best_hp.values)

{'units': 104}


In [48]:
model = tuner.get_best_models(num_models=1)[0]  # it will give the best model

C:\Users\divya\anaconda3\Lib\site-packages\keras\src\saving\saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [49]:
model.fit(X_train,y_train,batch_size=32,epochs=100,initial_epoch=6,validation_data=(X_test,y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.7769 - loss: 0.4697 - val_accuracy: 0.7662 - val_loss: 0.5078
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7785 - loss: 0.4596 - val_accuracy: 0.7727 - val_loss: 0.5072
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7785 - loss: 0.4534 - val_accuracy: 0.7727 - val_loss: 0.5018
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7801 - loss: 0.4495 - val_accuracy: 0.7727 - val_loss: 0.5037
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7818 - loss: 0.4459 - val_accuracy: 0.7662 - val_loss: 0.5059
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7834 - loss: 0.4432 - val_accuracy: 0.7662 - val_loss: 0.5067
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7834 - loss: 0.4404 - val_accuracy: 0.7597 - val_loss: 0.5089
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7818 - loss: 0.4382 - val_accuracy: 0.76

## How to select no. of layers?

In [50]:
def build_model(hp):

    model = Sequential()

    model.add(Dense(72,activation='relu',input_dim=8))

    for i in range(hp.Int('num_layers',min_value=1,max_value=10)):

        model.add(Dense(72,activation='relu'))

    model.add(Dense(1,activation='sigmoid'))

    model.compile(optimizer='rmsprop', loss='binary_crossentropy',metrics=['accuracy'])

    return model

In [51]:
tuner = kt.RandomSearch(build_model,
objective='val_accuracy',
max_trials=3,
directory='mydir',
project_name='num_layers')

Reloading Tuner from mydir\num_layers\tuner0.json


In [52]:
tuner.search(X_train,y_train, epochs=5,validation_data=(X_test,y_test))

In [53]:
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]  # num_trials 1 means give me the best 1 trail

print(best_hp.values)

{'num_layers': 3}


## 4. All in all one model

In [70]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

def build_model(hp):

    model = Sequential()

    counter = 0

    for i in range(hp.Int('num_layers', min_value=1, max_value=10)):

        if counter == 0:
            
            model.add(Dense(
                units=hp.Int('units' + str(i), min_value=8, max_value=128, step=8),
                activation=hp.Choice('activation' + str(i), values=['relu', 'tanh', 'sigmoid']),
                input_dim=8
            ))
            model.add(Dropout(hp.Choice('dropout'+str(i), values=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9])))

        else:

            model.add(Dense(
                units=hp.Int('units' + str(i), min_value=8, max_value=128, step=8),
                activation=hp.Choice('activation' + str(i), values=['relu', 'tanh', 'sigmoid'])
            ))
            model.add(Dropout(hp.Choice('dropout'+str(i), values=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9])))

        counter += 1

    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=hp.Choice('optimizer', values=['rmsprop', 'adam', 'sgd', 'nadam', 'adadelta']),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

"""This model allows Keras Tuner to search for:

Number of hidden layers

hp.Int('num_layers', min_value=1, max_value=10)

→ Between 1 and 10 hidden layers.

Number of neurons in each hidden layer

hp.Int('units0', ...)
hp.Int('units1', ...)
hp.Int('units2', ...)

→ Each layer can have 8, 16, 24, ..., 128 neurons.

Activation function for each hidden layer

hp.Choice(
    'activation0',
    ['relu', 'tanh', 'sigmoid']
)

→ Each layer can independently choose ReLU, Tanh, or Sigmoid.

Optimizer

hp.Choice(
    'optimizer',
    ['rmsprop', 'adam', 'sgd', 'nadam', 'adadelta']
)"""

"This model allows Keras Tuner to search for:\n\nNumber of hidden layers\n\nhp.Int('num_layers', min_value=1, max_value=10)\n\n→ Between 1 and 10 hidden layers.\n\nNumber of neurons in each hidden layer\n\nhp.Int('units0', ...)\nhp.Int('units1', ...)\nhp.Int('units2', ...)\n\n→ Each layer can have 8, 16, 24, ..., 128 neurons.\n\nActivation function for each hidden layer\n\nhp.Choice(\n    'activation0',\n    ['relu', 'tanh', 'sigmoid']\n)\n\n→ Each layer can independently choose ReLU, Tanh, or Sigmoid.\n\nOptimizer\n\nhp.Choice(\n    'optimizer',\n    ['rmsprop', 'adam', 'sgd', 'nadam', 'adadelta']\n)"

In [72]:
tuner = kt.RandomSearch(
    hypermodel=build_model,
    objective="val_accuracy",
    max_trials=3,
    overwrite=True,
    directory="my_dir",
    project_name="optimizer_search"
)

C:\Users\divya\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [73]:
tuner.search(X_train,y_train, epochs=5,validation_data=(X_test,y_test))

Trial 3 Complete [00h 00m 03s]
val_accuracy: 0.7272727489471436

Best val_accuracy So Far: 0.7337662577629089
Total elapsed time: 00h 00m 07s


In [74]:
tuner.get_best_hyperparameters(num_trials=1)[0].values

{'num_layers': 2,
 'units0': 88,
 'activation0': 'tanh',
 'dropout0': 0.9,
 'optimizer': 'nadam',
 'units1': 8,
 'activation1': 'relu',
 'dropout1': 0.1}

In [75]:
model = tuner.get_best_models(num_models=1)[0]  # it will give the best model

C:\Users\divya\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
C:\Users\divya\anaconda3\Lib\site-packages\keras\src\saving\saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'nadam', because it has 2 variables whereas the saved optimizer has 15 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [76]:
model.fit(X_train,y_train,batch_size=32,epochs=200,initial_epoch=6,validation_data=(X_test,y_test))

Epoch 7/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.5765 - loss: 0.6904 - val_accuracy: 0.7338 - val_loss: 0.5729
Epoch 8/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6287 - loss: 0.6847 - val_accuracy: 0.7468 - val_loss: 0.5618
Epoch 9/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6580 - loss: 0.6176 - val_accuracy: 0.7468 - val_loss: 0.5493
Epoch 10/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7068 - loss: 0.6074 - val_accuracy: 0.7662 - val_loss: 0.5413
Epoch 11/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7020 - loss: 0.6042 - val_accuracy: 0.7792 - val_loss: 0.5353
Epoch 12/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7101 - loss: 0.5471 - val_accuracy: 0.7662 - val_loss: 0.5309
Epoch 13/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7166 - loss: 0.5937 - val_accuracy: 0.7597 - val_loss: 0.5278
Epoch 14/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7166 - loss: 0.5599 - val_accuracy: 0.76

In [ ]:
# 1. Build a HyperModel
#           │
#           ▼
# 2. Define the Search Space
#           │
#           ▼
# 3. Create a Tuner
#           │
#           ▼
# 4. Search Best Hyperparameters
#           │
#           ▼
# 5. Get Best Hyperparameters
#           │
#           ▼
# 6. Build Final Model
#           │
#           ▼
# 7. Train Final Model
#           │
#           ▼
# 8. Evaluate Model